# SQLite Inspection

Detailed exploration notebook for `data/pystocks.sqlite`.

This notebook focuses on:
- schema and object inventory
- endpoint and series coverage
- dedupe/integrity checks for latest tables
- payload blob storage efficiency
- ingest run telemetry
- targeted deep dives by `conid`


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


def _resolve_repo_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for root in candidates:
        if (root / "data" / "pystocks.sqlite").exists():
            return root
    return cwd


ROOT = _resolve_repo_root()
DB_PATH = ROOT / "data" / "pystocks.sqlite"
if not DB_PATH.exists():
    raise FileNotFoundError(f"SQLite database not found at: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

print(f"Repo root: {ROOT}")
print(f"SQLite path: {DB_PATH}")


Repo root: /home/alex/Documents/pystocks
SQLite path: /home/alex/Documents/pystocks/data/pystocks.sqlite


In [2]:
def q(sql, params=None):
    params = params or []
    return pd.read_sql_query(sql, conn, params=params)


def table_columns(table_name):
    return q(f"PRAGMA table_info({table_name})")


def table_count(table_name):
    return int(q(f"SELECT COUNT(*) AS n FROM {table_name}").iloc[0]["n"])


In [3]:
tables = q("select * from profile_fees_fund_profile_fields")
print(len(tables))
tables

21


,conid,effective_at,asset_type,classification,distribution_details,domicile,fiscal_date,fund_category,fund_management_company,fund_manager_benchmark,fund_market_cap_focus,geographical_focus,inception_date,launch_opening_price,management_approach,management_expenses,manager_tenure,maturity_date,objective_type,portfolio_manager,redemption_charge_actual,redemption_charge_max,scheme,total_expense_ratio,total_net_assets_month_end_text,total_net_assets_month_end_num,total_net_assets_month_end_as_of_date
0,756733,2026-01-30,Equity,S&P 500 Index Funds,Paid,USA,9/30,Open-End Funds,State Street Global Advisors Trust Co,S&P 500 TR,Large-cap,United States of America,1993-01-22,1993-01-22,Passive,0.5250,NaN,NaN,S&P 500 Index Objective Funds,Undisclosed,NaN,NaN,US Mutual Fund Classification,0.0009,$708.92B (2026/01/30),7.089200e+11,2026-01-30
1,45540646,2026-01-30,Equity,S&P 500 Index Funds,Paid,USA,6/30,Open-End Funds,SSgA Funds Management Inc,S&P 500 TR,Large-cap,United States of America,2005-11-08,2005-11-08,Passive,0.9611,2014-01-01,NaN,S&P 500 Index Objective Funds,Schneider/Law/Rabinovich,NaN,NaN,US Mutual Fund Classification,0.0002,$106.21B (2026/01/30),1.062100e+11,2026-01-30
2,89008395,2026-01-30,Other,NaN,Retained,Jersey,12/31,NaN,Deutsche Bank Ag (London Branch),Index is not available on Lipper Database,NaN,EuroZone,NaN,2015-11-09,Passive,NaN,NaN,NaN,NaN,undisclosed,0.0,NaN,NaN,NaN,€20.86M (2026/01/30),2.086000e+07,2026-01-30
3,689039014,2026-01-30,Commodity,NaN,Paid,Canada,12/31,NaN,BMO Asset Management Inc,Index is not provided by Management Company,NaN,United States of America,NaN,2024-03-08,Active,NaN,NaN,NaN,NaN,Robert Bechard,NaN,NaN,NaN,NaN,CAD289.15M (2026/01/30),NaN,2026-01-30
4,689038580,2026-01-30,Commodity,NaN,Paid,Canada,12/31,NaN,BMO Asset Management Inc,Index is not provided by Management Company,NaN,United States of America,NaN,2024-03-08,Active,NaN,NaN,NaN,NaN,Robert Bechard,NaN,NaN,NaN,NaN,CAD1.7B (2026/01/30),NaN,2026-01-30
5,89384980,2026-01-30,Equity,NaN,Retained,Ireland,3/31,NaN,State Street Global Advisors Europe Ltd,MSCI EM (Emerging Markets) NR USD,Large-cap,Global Emerging Markets,NaN,2011-05-13,Passive,NaN,NaN,NaN,NaN,Undisclosed,NaN,0.00,NaN,NaN,$1.66B (2026/01/30),1.660000e+09,2026-01-30
6,565109827,2026-01-30,Equity,NaN,Retained,Ireland,6/30,NaN,Global X Management Company (Europe) Ltd,Solactive Global Uranium & Nuclear Components ...,Broad Market,Global,NaN,2022-04-20,Passive,NaN,NaN,NaN,NaN,Company Managed,0.0,0.03,NaN,NaN,$680.4M (2026/01/30),6.804000e+08,2026-01-30
7,802787801,2026-01-30,Equity,NaN,Retained,Ireland,12/31,NaN,DWS Investment SA,NASDAQ 100 CR,Large-cap,United States of America,NaN,2025-07-09,Passive,NaN,NaN,NaN,NaN,Team Managed,0.0,NaN,NaN,NaN,$2.72M (2026/01/30),2.720000e+06,2026-01-30
8,75961307,2026-01-30,Equity,NaN,Retained,Ireland,7/31,NaN,BlackRock Asset Management Ireland Ltd,EURO STOXX 50 NR EUR,Large-cap,EuroZone,NaN,2010-01-26,Passive,NaN,2010-01-26,NaN,NaN,Company Managed,NaN,NaN,NaN,NaN,€7.09B (2026/01/30),7.090000e+09,2026-01-30
9,703680889,2027-12-31,Bond,NaN,Paid,Luxembourg,9/30,NaN,Amundi Luxembourg SA,Index is not available on Lipper Database,NaN,Germany,NaN,2024-04-25,Passive,NaN,NaN,2027-12-31,NaN,Undisclosed,NaN,0.03,NaN,NaN,€6.48M (2026/01/30),6.480000e+06,2026-01-30


## Object Inventory


In [17]:
q("select * from landing_snapshots limit 50")#.columns

DatabaseError: Execution failed on sql 'select * from landing_snapshots limit 50': no such table: landing_snapshots

In [7]:
objects = q(
    """
    SELECT *
    FROM sqlite_master
    where type = 'table'
    ORDER BY type, name
    """
)

display(objects[["type", "name"]])

tables = objects.loc[objects["type"] == "table", "name"].tolist()
views = objects.loc[objects["type"] == "view", "name"].tolist()

print(f"tables: {len(tables)}")
print(f"views:  {len(views)}")


,type,name
0,table,dividends_events_series_latest
1,table,dividends_events_series_raw
2,table,dividends_industry_metrics
3,table,dividends_snapshots
4,table,endpoint_scalar_extras
5,table,esg_nodes
6,table,esg_snapshots
7,table,holdings_bucket_weights
8,table,holdings_geographic_weights
9,table,holdings_snapshots


tables: 48
views:  0


In [8]:
pragma_names = [
    "journal_mode",
    "synchronous",
    "foreign_keys",
    "temp_store",
    "wal_autocheckpoint",
]

pragma_rows = []
for name in pragma_names:
    value = q(f"PRAGMA {name}").iloc[0, 0]
    pragma_rows.append({"pragma": name, "value": value})

display(pd.DataFrame(pragma_rows))

if "schema_meta" in tables:
    display(q("SELECT * FROM schema_meta ORDER BY applied_at DESC"))
else:
    print("schema_meta table not found")


,pragma,value
0,journal_mode,wal
1,synchronous,2
2,foreign_keys,0
3,temp_store,0
4,wal_autocheckpoint,1000


,schema_version,applied_at
0,1,2026-02-25T13:08:13.897567+00:00


In [9]:
counts = []
for table in tables:
    counts.append({"table": table, "rows": table_count(table)})

counts_df = pd.DataFrame(counts).sort_values(["rows", "table"], ascending=[False, True]).reset_index(drop=True)
display(counts_df)


,table,rows
0,price_chart_series_latest,48234
1,price_chart_series_raw,48234
2,products,20801
3,sentiment_search_series_latest,10038
4,sentiment_search_series_raw,10038
5,lipper_ratings_values,1497
6,ratios_metrics,536
7,holdings_bucket_weights,467
8,performance_metrics,445
9,holdings_top10_conids,363


## Endpoint Coverage


In [ ]:
snapshot_tables = sorted([t for t in tables if t.endswith("_snapshots")])

snapshot_rows = []
for table in snapshot_tables:
    row = q(
        f"""
        SELECT
            COUNT(*) AS snapshot_rows,
            COUNT(DISTINCT conid) AS conids,
            MIN(effective_at) AS min_effective_at,
            MAX(effective_at) AS max_effective_at,
            MAX(observed_at) AS last_observed_at
        FROM {table}
        """
    ).iloc[0]
    snapshot_rows.append(
        {
            "endpoint": table.replace("_snapshots", ""),
            "table": table,
            "snapshot_rows": int(row["snapshot_rows"]),
            "conids": int(row["conids"]),
            "min_effective_at": row["min_effective_at"],
            "max_effective_at": row["max_effective_at"],
            "last_observed_at": row["last_observed_at"],
        }
    )

snapshot_df = pd.DataFrame(snapshot_rows).sort_values(["snapshot_rows", "endpoint"], ascending=[False, True]).reset_index(drop=True)
display(snapshot_df)


In [ ]:
series_raw_tables = sorted([t for t in tables if t.endswith("_series_raw")])
series_latest_tables = sorted([t for t in tables if t.endswith("_series_latest")])

series_rows = []
for raw_table in series_raw_tables:
    base = raw_table[: -len("_series_raw")]
    latest_table = f"{base}_series_latest"
    raw_n = table_count(raw_table)
    latest_n = table_count(latest_table) if latest_table in series_latest_tables else 0
    ratio = (raw_n / latest_n) if latest_n else None
    series_rows.append(
        {
            "series": base,
            "raw_table": raw_table,
            "latest_table": latest_table,
            "raw_rows": raw_n,
            "latest_rows": latest_n,
            "raw_to_latest_ratio": ratio,
        }
    )

series_df = pd.DataFrame(series_rows).sort_values("series").reset_index(drop=True)
display(series_df)


## Storage and Telemetry


In [ ]:
if "raw_payload_blobs" in tables:
    blob_stats = q(
        """
        SELECT
            COUNT(*) AS blob_rows,
            SUM(raw_size_bytes) AS raw_bytes,
            SUM(compressed_size_bytes) AS compressed_bytes,
            ROUND(1.0 * SUM(compressed_size_bytes) / NULLIF(SUM(raw_size_bytes), 0), 4) AS compressed_to_raw_ratio,
            MIN(created_at) AS first_created_at,
            MAX(created_at) AS last_created_at
        FROM raw_payload_blobs
        """
    )
    display(blob_stats)
else:
    print("raw_payload_blobs table not found")


In [ ]:
if "ingest_runs" in tables:
    display(
        q(
            """
            SELECT
                run_id,
                run_started_at,
                run_finished_at,
                total_targeted_conids,
                processed_conids,
                saved_snapshots,
                inserted_events,
                overwritten_events,
                unchanged_events,
                series_raw_rows_written,
                series_latest_rows_upserted,
                auth_retries,
                aborted
            FROM ingest_runs
            ORDER BY run_id DESC
            LIMIT 20
            """
        )
    )

    if "ingest_run_endpoint_rollups" in tables:
        latest_run = q("SELECT run_id FROM ingest_runs ORDER BY run_id DESC LIMIT 1")
        if not latest_run.empty:
            run_id = int(latest_run.iloc[0]["run_id"])
            print(f"Endpoint rollups for run_id={run_id}")
            display(
                q(
                    """
                    SELECT
                        endpoint,
                        call_count,
                        useful_payload_count,
                        useful_payload_rate,
                        status_2xx,
                        status_4xx,
                        status_5xx,
                        status_other
                    FROM ingest_run_endpoint_rollups
                    WHERE run_id = ?
                    ORDER BY call_count DESC, endpoint
                    """,
                    [run_id],
                )
            )
else:
    print("ingest_runs table not found")


## Integrity Checks


In [ ]:
duplicate_checks = [
    ("price_chart_series_latest", "conid, trade_date"),
    ("sentiment_search_series_latest", "conid, datetime_ms"),
    (
        "ownership_trade_log_series_latest",
        "conid, trade_date, action, party, source, insider, shares, value, holding",
    ),
    (
        "dividends_events_series_latest",
        "conid, event_date, amount, currency, event_type, declaration_date, record_date, payment_date, description",
    ),
]

dup_rows = []
for table, key_cols in duplicate_checks:
    if table not in tables:
        dup_rows.append({"table": table, "duplicate_key_groups": None})
        continue

    dup_n = int(
        q(
            f"""
            SELECT COUNT(*) AS n
            FROM (
                SELECT {key_cols}, COUNT(*) AS c
                FROM {table}
                GROUP BY {key_cols}
                HAVING c > 1
            )
            """
        ).iloc[0]["n"]
    )
    dup_rows.append({"table": table, "duplicate_key_groups": dup_n})

display(pd.DataFrame(dup_rows))


In [ ]:
if "products" in tables:
    display(
        q(
            """
            SELECT
                COALESCE(last_status_fundamentals, '<null>') AS fundamentals_status,
                COUNT(*) AS products
            FROM products
            GROUP BY fundamentals_status
            ORDER BY products DESC, fundamentals_status
            """
        )
    )

    display(
        q(
            """
            SELECT
                COUNT(*) AS total_products,
                SUM(CASE WHEN last_scraped_fundamentals IS NOT NULL THEN 1 ELSE 0 END) AS scraped_once,
                MAX(last_scraped_fundamentals) AS latest_scrape_day
            FROM products
            """
        )
    )


## Conid Deep Dive


In [ ]:
TARGET_CONID = None

if snapshot_tables:
    union_sql = " UNION ALL ".join(
        [f"SELECT conid, '{table}' AS endpoint_table FROM {table}" for table in snapshot_tables]
    )
    coverage = q(
        f"""
        WITH endpoint_events AS (
            {union_sql}
        )
        SELECT
            conid,
            COUNT(DISTINCT endpoint_table) AS endpoint_tables,
            COUNT(*) AS snapshot_rows
        FROM endpoint_events
        GROUP BY conid
        ORDER BY endpoint_tables DESC, snapshot_rows DESC, conid
        LIMIT 20
        """
    )
    display(coverage)

    if TARGET_CONID is None and not coverage.empty:
        TARGET_CONID = str(coverage.iloc[0]["conid"])

print(f"TARGET_CONID: {TARGET_CONID}")


In [ ]:
if TARGET_CONID:
    if "products" in tables:
        display(q("SELECT * FROM products WHERE conid = ?", [TARGET_CONID]))

    if snapshot_tables:
        timeline_union = " UNION ALL ".join(
            [
                f"SELECT '{table}' AS endpoint_table, effective_at, observed_at, payload_hash FROM {table} WHERE conid = ?"
                for table in snapshot_tables
            ]
        )
        params = [TARGET_CONID] * len(snapshot_tables)
        timeline = q(
            f"""
            {timeline_union}
            ORDER BY observed_at DESC, endpoint_table
            """,
            params,
        )
        display(timeline.head(300))

        hash_changes = q(
            f"""
            WITH target_snapshots AS (
                {timeline_union}
            )
            SELECT
                endpoint_table,
                COUNT(*) AS rows,
                COUNT(DISTINCT payload_hash) AS distinct_payload_hashes,
                MIN(effective_at) AS min_effective_at,
                MAX(effective_at) AS max_effective_at
            FROM target_snapshots
            GROUP BY endpoint_table
            ORDER BY rows DESC, endpoint_table
            """,
            params,
        )
        display(hash_changes)


In [ ]:
if TARGET_CONID:
    series_queries = [
        (
            "price_chart_series_latest",
            "SELECT conid, trade_date, close, price, effective_at, observed_at FROM price_chart_series_latest WHERE conid = ? ORDER BY trade_date DESC LIMIT 50",
        ),
        (
            "sentiment_search_series_latest",
            "SELECT conid, trade_date, datetime_ms, sscore, sdelta, sbuzz, effective_at, observed_at FROM sentiment_search_series_latest WHERE conid = ? ORDER BY datetime_ms DESC LIMIT 50",
        ),
        (
            "dividends_events_series_latest",
            "SELECT conid, event_date, event_type, amount, currency, declaration_date, payment_date, effective_at FROM dividends_events_series_latest WHERE conid = ? ORDER BY event_date DESC LIMIT 50",
        ),
        (
            "ownership_trade_log_series_latest",
            "SELECT conid, trade_date, action, party, shares, value, holding, effective_at FROM ownership_trade_log_series_latest WHERE conid = ? ORDER BY trade_date DESC LIMIT 50",
        ),
    ]

    for table, sql in series_queries:
        if table in tables:
            print(table)
            display(q(sql, [TARGET_CONID]))


## Ad Hoc SQL

Use `q("...")` for additional checks, for example:
- `q("SELECT * FROM ratios_metrics LIMIT 50")`
- `q("SELECT endpoint, SUM(call_count) FROM ingest_run_endpoint_rollups GROUP BY endpoint ORDER BY 2 DESC")`

When finished, close the connection in a cell with `conn.close()`.
